In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
# Loading in US car accident dataset
car_data = pd.read_csv("US_Accidents_March23_sampled_500k.csv")
car_data

,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-2047758,Source2,2,2019-06-12 10:10:56,2019-06-12 10:55:58,30.641211,-91.153481,NaN,NaN,0.000,...,False,False,False,False,True,False,Day,Day,Day,Day
1,A-4694324,Source1,2,2022-12-03 23:37:14.000000000,2022-12-04 01:56:53.000000000,38.990562,-77.399070,38.990037,-77.398282,0.056,...,False,False,False,False,False,False,Night,Night,Night,Night
2,A-5006183,Source1,2,2022-08-20 13:13:00.000000000,2022-08-20 15:22:45.000000000,34.661189,-120.492822,34.661189,-120.492442,0.022,...,False,False,False,False,True,False,Day,Day,Day,Day
3,A-4237356,Source1,2,2022-02-21 17:43:04,2022-02-21 19:43:23,43.680592,-92.993317,43.680574,-92.972223,1.054,...,False,False,False,False,False,False,Day,Day,Day,Day
4,A-6690583,Source1,2,2020-12-04 01:46:00,2020-12-04 04:13:09,35.395484,-118.985176,35.395476,-118.985995,0.046,...,False,False,False,False,False,False,Night,Night,Night,Night
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499995,A-6077227,Source1,2,2021-12-15 07:30:00,2021-12-15 07:50:30,45.522510,-123.084104,45.520225,-123.084211,0.158,...,False,False,False,False,False,False,Night,Day,Day,Day
499996,A-6323243,Source1,2,2021-12-19 16:25:00,2021-12-19 17:40:37,26.702570,-80.111169,26.703141,-80.111133,0.040,...,False,False,False,False,False,False,Day,Day,Day,Day
499997,A-3789256,Source1,2,2022-04-13 19:28:29,2022-04-13 21:33:44,34.561862,-112.259620,34.566822,-112.267150,0.549,...,False,False,False,False,True,False,Night,Night,Day,Day
499998,A-7030381,Source1,3,2020-05-15 17:20:56,2020-05-15 17:50:56,38.406680,-78.619310,38.406680,-78.619310,0.000,...,False,False,False,False,True,False,Day,Day,Day,Day


In [3]:
#Simplifying down the weather related columns we need to answer our research question
weather_columns = ["Temperature(F)",
                   "Humidity(%)",
                   "Visibility(mi)",
                   "Wind_Speed(mph)",
                   "Precipitation(in)",
                   "Weather_Condition"]
weather_columns

['Temperature(F)',
 'Humidity(%)',
 'Visibility(mi)',
 'Wind_Speed(mph)',
 'Precipitation(in)',
 'Weather_Condition']

In [4]:
#Cleaning Data up to only display weather related columns plus severity
weather_data_df = car_data[["ID", "Severity", "Start_Time"] + weather_columns].copy()
weather_data_df.head(5)

,ID,Severity,Start_Time,Temperature(F),Humidity(%),Visibility(mi),Wind_Speed(mph),Precipitation(in),Weather_Condition
0,A-2047758,2,2019-06-12 10:10:56,77.0,62.0,10.0,5.0,0.0,Fair
1,A-4694324,2,2022-12-03 23:37:14.000000000,45.0,48.0,10.0,5.0,0.0,Fair
2,A-5006183,2,2022-08-20 13:13:00.000000000,68.0,73.0,10.0,13.0,0.0,Fair
3,A-4237356,2,2022-02-21 17:43:04,27.0,86.0,10.0,15.0,0.0,Wintry Mix
4,A-6690583,2,2020-12-04 01:46:00,42.0,34.0,10.0,0.0,0.0,Fair


In [5]:
# Preserve missing precipitation values because we dont want to just plug in the average/mean value as it'll make our data biased
weather_data_df["Precipitation_Missing"] = (
    weather_data_df["Precipitation(in)"].isna()
)

# See all missing values across all weather variables
missing_values_table = pd.DataFrame({
    "Missing Count": weather_data_df[weather_columns].isna().sum(),
    "Missing Percent": (weather_data_df[weather_columns].isna().mean() * 100).round(2)})

missing_values_table #Visualize all missing values for our data 

,Missing Count,Missing Percent
Temperature(F),10466,2.09
Humidity(%),11130,2.23
Visibility(mi),11291,2.26
Wind_Speed(mph),36987,7.40
Precipitation(in),142616,28.52
Weather_Condition,11101,2.22


In [6]:
numerical_weather_cols = [
    "Temperature(F)",
    "Humidity(%)",
    "Visibility(mi)",
    "Wind_Speed(mph)",
    "Precipitation(in)"]

# Summary statistics and percentiles of our numerical datatype weather data
weather_summary = weather_data_df[
    numerical_weather_cols].describe()

weather_summary


,Temperature(F),Humidity(%),Visibility(mi),Wind_Speed(mph),Precipitation(in)
count,489534.000000,488870.000000,488709.000000,463013.000000,357384.000000
mean,61.646254,64.834921,9.091540,7.681347,0.008289
std,19.000133,22.826158,2.708083,5.431361,0.101865
min,-77.800000,1.000000,0.000000,0.000000,0.000000
25%,49.000000,48.000000,10.000000,4.600000,0.000000
50%,64.000000,67.000000,10.000000,7.000000,0.000000
75%,76.000000,84.000000,10.000000,10.400000,0.000000
max,207.000000,100.000000,130.000000,822.800000,10.130000


In [7]:

# Define thresholds for questionable/extreme weather values
outlier_boundaries = {
    "Temperature(F)": (-60, 140),
    "Humidity(%)": (0, 100),
    "Visibility(mi)": (0, 30),
    "Wind_Speed(mph)": (0, 150),
    "Precipitation(in)": (0, None)}

# Count values outside each threshold
for col, (lower, upper) in outlier_boundaries.items():
    below = (weather_data_df[col] < lower).sum()
    if upper is not None:
        above = (weather_data_df[col] > upper).sum()
    else:
        above = 0
        
    print(f"{col}: {below + above} extreme values")


Temperature(F): 3 extreme values
Humidity(%): 0 extreme values
Visibility(mi): 405 extreme values
Wind_Speed(mph): 3 extreme values
Precipitation(in): 0 extreme values


In [8]:
# Here we are replacing extreme temperature values with NaN
weather_data_df.loc[weather_data_df["Temperature(F)"] < -60, "Temperature(F)"] = np.nan
weather_data_df.loc[weather_data_df["Temperature(F)"] > 140, "Temperature(F)"] = np.nan

# Replace extreme wind speeds with NaN
weather_data_df.loc[weather_data_df["Wind_Speed(mph)"] > 150, "Wind_Speed(mph)"] = np.nan

# Here we are checking the updated values
weather_data_df[["Temperature(F)", "Wind_Speed(mph)"]].describe()

,Temperature(F),Wind_Speed(mph)
count,489531.000000,463010.000000
mean,61.645967,7.678632
std,18.997040,5.277626
min,-35.000000,0.000000
25%,49.000000,4.600000
50%,64.000000,7.000000
75%,76.000000,10.400000
max,140.000000,142.000000


In [9]:

# Here we are counting how often each weather condition appears
weather_counts = weather_data_df[
    "Weather_Condition"].value_counts(dropna=False)

weather_counts.head(30)

Weather_Condition
Fair                       166053
Mostly Cloudy               65398
Cloudy                      53107
Clear                       52379
Partly Cloudy               45125
Overcast                    24901
Light Rain                  22718
Scattered Clouds            13030
NaN                         11101
Light Snow                   8233
Fog                          6445
Rain                         5501
Haze                         4967
Fair / Windy                 2361
Heavy Rain                   2133
Light Drizzle                1474
Thunder in the Vicinity      1186
Mostly Cloudy / Windy        1123
Cloudy / Windy               1080
T-Storm                      1052
Snow                          982
Thunder                       925
Light Rain with Thunder       819
Smoke                         805
Wintry Mix                    777
Partly Cloudy / Windy         654
Heavy T-Storm                 647
Light Rain / Windy            507
Light Snow / Windy            

In [10]:

def group_weather(weather_condition):
    if pd.isna(weather_condition):
        return "Unknown"

    weather_condition = weather_condition.lower().strip() 
    #coverting all text to lowercase and removing extra spaces before after word

    if "n/a" in weather_condition: 
        return "Unknown" # Grouping togehter unkown weather conditions

    elif "thunder" in weather_condition or "t-storm" in weather_condition:
        return "Thunderstorm" # grouping together thunder storms

    elif any(word in weather_condition for word in [
        "snow", "ice", "sleet", "wintry",
        "freezing", "hail"]):
        return "Snow/Ice" # grouping together snow and ice

    elif any(word in weather_condition for word in [
        "rain", "drizzle", "shower"]):
        return "Rain" # Again we are grouping together rain terms

    elif any(word in weather_condition for word in [
        "fog", "haze", "mist", "smoke",
        "dust", "sand", "ash"]):
        return "Fog/Haze" # grouping together fog and haze words

    elif any(word in weather_condition for word in [
        "windy", "tornado", "squall"]): 
        return "Windy" #Same thing for windy words

    elif any(word in weather_condition for word in [
        "cloud", "overcast"]):
        return "Cloudy" # Grouping cloudy terms

    elif "fair" in weather_condition or "clear" in weather_condition:
        return "Clear" # Grouping clear weather words once again

    else:
        return "Other"


# Applying our function to all weather conditions
weather_data_df["Weather_Category"] = (
    weather_data_df["Weather_Condition"].apply(group_weather))

weather_data_df["Weather_Category"].value_counts()


Weather_Category
Clear           218432
Cloudy          201567
Rain             33123
Fog/Haze         13144
Snow/Ice         11481
Unknown          11308
Thunderstorm      5723
Windy             5222
Name: count, dtype: int64

In [11]:
# Convert car accident start times to datetime format
weather_data_df["Start_Time"] = pd.to_datetime(
    weather_data_df["Start_Time"], format="mixed",errors="coerce")

In [12]:

# Check missing values in our weather variables
missing_summary = pd.DataFrame({
    "Missing Value Count": weather_data_df[numerical_weather_cols + ["Weather_Condition"]].isna().sum(),
    "Missing Percentage": (weather_data_df[numerical_weather_cols + ["Weather_Condition"]].isna().mean() * 100).round(2)})

missing_summary

,Missing Value Count,Missing Percentage
Temperature(F),10469,2.09
Humidity(%),11130,2.23
Visibility(mi),11291,2.26
Wind_Speed(mph),36990,7.40
Precipitation(in),142616,28.52
Weather_Condition,11101,2.22


In [13]:
# Final columns needed for our dataset
final_weather_columns = [
    "ID",
    "Severity",
    "Start_Time",
    "Temperature(F)",
    "Humidity(%)",
    "Visibility(mi)",
    "Wind_Speed(mph)",
    "Precipitation(in)",
    "Weather_Condition",
    "Precipitation_Missing",
    "Weather_Category"
]

weather_data_df = weather_data_df[final_weather_columns].copy()

# Final check of our cleaned weather dataset
print(f"Total rows: {len(weather_data_df)}")
print(f"Total columns: {len(weather_data_df.columns)}")

weather_data_df.head(5)


Total rows: 500000
Total columns: 11


,ID,Severity,Start_Time,Temperature(F),Humidity(%),Visibility(mi),Wind_Speed(mph),Precipitation(in),Weather_Condition,Precipitation_Missing,Weather_Category
0,A-2047758,2,2019-06-12 10:10:56,77.0,62.0,10.0,5.0,0.0,Fair,False,Clear
1,A-4694324,2,2022-12-03 23:37:14,45.0,48.0,10.0,5.0,0.0,Fair,False,Clear
2,A-5006183,2,2022-08-20 13:13:00,68.0,73.0,10.0,13.0,0.0,Fair,False,Clear
3,A-4237356,2,2022-02-21 17:43:04,27.0,86.0,10.0,15.0,0.0,Wintry Mix,False,Snow/Ice
4,A-6690583,2,2020-12-04 01:46:00,42.0,34.0,10.0,0.0,0.0,Fair,False,Clear


In [14]:

# Saving cleaned weather dataset
weather_data_df.to_csv("weather_cleaned.csv", index=False)
weather_data_df.shape

(500000, 11)